# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [14]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [15]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


In [16]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [17]:
# Use esta célula para criar sua análise exploratória.

colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

df[colunas_numericas].corr()

,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


In [18]:
variavel_x = "taxa_abandono_carrinho_pct"

fig = px.scatter(
    df,
    x=variavel_x,
    y="taxa_conversao_pct",
    trendline="ols",
    title="Relação entre abandono do carrinho e taxa de conversão",
    labels={
        "taxa_abandono_carrinho_pct": "Taxa de abandono do carrinho (%)",
        "taxa_conversao_pct": "Taxa de conversão (%)"
    }
)

fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

Escolhi a taxa de abandono do carrinho e a profundidade média de scroll para fazer a análise de sensibilidade.

A taxa de abandono do carrinho teve a maior relação com a taxa de conversão, com correlação de -0,643. Pelo gráfico, dá para perceber que, quando o abandono do carrinho aumenta, a taxa de conversão tende a diminuir.

A profundidade média de scroll teve correlação positiva de 0,485 com a conversão. Isso mostra que, quando os usuários navegam mais pela página, a conversão tende a ser maior.

Já o tempo até o primeiro clique teve uma relação mais fraca com a conversão, com correlação de -0,229. Por isso, não escolhi essa variável.


## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [19]:
X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

,métrica,valor
0,MAE,0.276
1,RMSE,0.344


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

O MAE do modelo foi 0,276 e o RMSE foi 0,344.

Isso significa que, em média, as previsões do modelo ficaram cerca de 0,28 ponto percentual diferentes da taxa de conversão real. Como a taxa de conversão varia aproximadamente entre 4% e 8%, esse erro é relativamente baixo.

O RMSE ficou um pouco maior que o MAE porque ele dá mais peso para os erros mais altos. Mesmo assim, o resultado mostra que o modelo consegue estimar a taxa de conversão de forma razoável.


## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [20]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [21]:
# Preencha com duas variáveis escolhidas na Parte 1.
# Use exatamente os nomes que aparecem em features.

variaveis_escolhidas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct"
]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258


In [22]:
fig = px.bar(
    tabela_sensibilidade,
    x="variável",
    y="índice_sensibilidade",
    title="Comparação dos índices de sensibilidade",
    labels={
        "variável": "Variável",
        "índice_sensibilidade": "Índice de sensibilidade"
    }
)

fig.show()

Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

A taxa de abandono do carrinho foi a variável que teve maior impacto na taxa de conversão.

Quando ela aumentou 10%, passando de 47,546% para 52,300%, a taxa de conversão prevista caiu de 5,869% para 5,582%. Isso representou uma variação de -4,891% na saída e um índice de sensibilidade de -0,489.

Já a profundidade média de scroll, aumentando 10%, fez a taxa de conversão subir de 5,869% para 6,020%. A variação da saída foi de 2,578% e o índice de sensibilidade foi 0,258.

Como o valor absoluto de -0,489 é maior que 0,258, o abandono do carrinho tem mais impacto na conversão. Por isso, a prioridade seria reduzir o abandono do carrinho.


## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

A recomendação é priorizar melhorias na etapa do carrinho e checkout para reduzir o abandono. Por exemplo, a interface pode deixar o valor final mais claro, diminuir etapas para finalizar a compra e facilitar o preenchimento de dados e pagamento.

Na análise de sensibilidade, um aumento de 10% no abandono do carrinho reduziu a conversão prevista de 5,869% para 5,582%, uma queda de 4,891%. O índice de sensibilidade foi -0,489, maior em valor absoluto do que o da profundidade de scroll, que foi 0,258. Por isso, reduzir o abandono do carrinho deve ser a prioridade.

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

Uma limitação é que a análise mostra relações entre as variáveis, mas não prova que uma causa diretamente a outra. Por exemplo, um abandono maior pode acontecer também por fatores que não estão na base, como preço, frete, estoque ou problemas no pagamento.

Além disso, o modelo foi feito com dados simulados e considera as outras variáveis constantes durante a análise de sensibilidade. Na prática, seria importante validar essa decisão com dados reais e, de preferência, com um teste A/B.

## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [23]:
rng_sim = np.random.default_rng(2026)

n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng_sim.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng_sim.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng_sim.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])

previsoes = amostras_design @ coeficientes

pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

,0
count,1000.000
mean,5.848
std,0.391
min,4.647
10%,5.339
25%,5.574
50%,5.853
75%,6.123
90%,6.338
max,6.993


In [24]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

In [25]:
amostras_melhoria = amostras.copy()

amostras_melhoria["taxa_abandono_carrinho_pct"] = (
    amostras_melhoria["taxa_abandono_carrinho_pct"] * 0.90
).clip(lower=25, upper=75)

amostras_melhoria_design = np.column_stack([
    np.ones(len(amostras_melhoria)),
    amostras_melhoria[features].to_numpy()
])

previsoes_melhoria = amostras_melhoria_design @ coeficientes

comparacao_monte_carlo = pd.DataFrame({
    "cenário": ["Atual", "Abandono 10% menor"],
    "média da conversão prevista (%)": [
        previsoes.mean(),
        previsoes_melhoria.mean()
    ],
    "p10 (%)": [
        np.percentile(previsoes, 10),
        np.percentile(previsoes_melhoria, 10)
    ],
    "p90 (%)": [
        np.percentile(previsoes, 90),
        np.percentile(previsoes_melhoria, 90)
    ]
})

comparacao_monte_carlo["ganho médio vs atual (p.p.)"] = (
    comparacao_monte_carlo["média da conversão prevista (%)"]
    - comparacao_monte_carlo.loc[0, "média da conversão prevista (%)"]
)

comparacao_monte_carlo

,cenário,média da conversão prevista (%),p10 (%),p90 (%),ganho médio vs atual (p.p.)
0,Atual,5.848,5.339,6.338,0.000
1,Abandono 10% menor,6.136,5.665,6.598,0.288


In [26]:
dados_cenarios = pd.DataFrame({
    "cenário": (
        ["Atual"] * len(previsoes)
        + ["Abandono 10% menor"] * len(previsoes_melhoria)
    ),
    "taxa_conversao_prevista_pct": np.concatenate([
        previsoes,
        previsoes_melhoria
    ])
})

fig = px.box(
    dados_cenarios,
    x="cenário",
    y="taxa_conversao_prevista_pct",
    title="Conversão simulada: cenário atual vs. redução do abandono",
    labels={
        "cenário": "Cenário",
        "taxa_conversao_prevista_pct": "Taxa de conversão prevista (%)"
    }
)

fig.show()

Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

A simulação mostra que existe incerteza nas variáveis de entrada, então a taxa de conversão pode mudar de um cenário para outro. No cenário atual, a média da conversão prevista foi de 5,848%.

Também foi simulado um cenário com redução de 10% no abandono do carrinho. Nesse caso, a média da conversão prevista aumentou para 6,136%, um ganho médio de 0,288 ponto percentual. O percentil de 10% aumentou de 5,339% para 5,665%, e o percentil de 90% subiu de 6,338% para 6,598%.

Isso reforça a recomendação de priorizar a redução do abandono do carrinho. Mesmo considerando a incerteza das outras variáveis, o cenário de melhoria apresentou resultados melhores. Ainda assim, seria importante validar essa mudança com dados reais ou com um teste A/B.

## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.